In [ ]:
import pandas as pd

In [4]:
ans_df = pd.read_csv('./dados/dados_plano_saude.csv', sep=';', encoding='utf-8')

In [10]:
competencia_col = ans_df.columns[0]
ans_df = ans_df[ans_df[competencia_col].str.startswith('Jun/')]

ans_df['ano'] = ans_df[competencia_col].str.split('/').str[1].astype(int) + 2000

id_vars = [competencia_col, 'ano']
value_vars = [col for col in ans_df.columns if col not in id_vars and col != 'SEM_IDENTIFICACAO' and col != 'TOTAL']
ans_long = ans_df.melt(id_vars=id_vars, value_vars=value_vars, var_name='uf', value_name='beneficiarios')

pop_df = pd.read_excel('../IBGE/populacao_PNAD.xlsx', sheet_name='Tabela')

state_mapping = {
    'Rondônia': 'RO',
    'Acre': 'AC',
    'Amazonas': 'AM',
    'Roraima': 'RR',
    'Pará': 'PA',
    'Amapá': 'AP',
    'Tocantins': 'TO',
    'Maranhão': 'MA',
    'Piauí': 'PI',
    'Ceará': 'CE',
    'Rio Grande do Norte': 'RN',
    'Paraíba': 'PB',
    'Pernambuco': 'PE',
    'Alagoas': 'AL',
    'Sergipe': 'SE',
    'Bahia': 'BA',
    'Minas Gerais': 'MG',
    'Espírito Santo': 'ES',
    'Rio de Janeiro': 'RJ',
    'São Paulo': 'SP',
    'Paraná': 'PR',
    'Santa Catarina': 'SC',
    'Rio Grande do Sul': 'RS',
    'Mato Grosso do Sul': 'MS',
    'Mato Grosso': 'MT',
    'Goiás': 'GO',
    'Distrito Federal': 'DF'
}

pop_df['uf'] = pop_df['Unidade da Federação'].map(state_mapping)

pop_long = pop_df.melt(id_vars=['uf'], value_vars=[2019, 2020, 2021, 2022, 2023, 2024], var_name='ano', value_name='populacao')
pop_long['ano'] = pop_long['ano'].astype(int)
pop_long['populacao'] = pop_long['populacao'] * 1000 

proj_df = pd.read_excel('../IBGE/projecao_2025.xlsx', sheet_name='Tabela')
proj_df['uf'] = proj_df['Unidade da Federação'].map(state_mapping)
proj_df['ano'] = 2025
proj_df['populacao'] = proj_df['total']

pop_long = pd.concat([pop_long, proj_df[['uf', 'ano', 'populacao']]], ignore_index=True)

merged = pd.merge(pop_long, ans_long, on=['uf', 'ano'], how='left')

merged['beneficiarios'] = merged['beneficiarios'].fillna(0)

result = merged[['ano', 'uf', 'populacao', 'beneficiarios']]

result['dependentes_do_sus'] = result['populacao'] - result['beneficiarios']
result['percentual_dependentes'] = (result['dependentes_do_sus'] / result['populacao']) 

result.to_excel('./dados/merged_ans_populacao.xlsx', index=False)